# 2-E (optional after class): Run the whole division-of-labor diagram in Colab, then see how it goes to the cloud

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/en/notebooks/2-E_deploy.ipynb)

> 📍 **Matching slides**: Session 2, Stop 4, slide **37** "The Homework Helper's full division of labor", slide **48** "The division of labor in the cloud: where each box lives", and "Supplementary: moving to the cloud", slides **47-55** (self-study after class)

## 🎯 The mini project for these two sessions: the Homework Helper

The **Homework Helper** is a small program that helps check Python homework. You give it a piece of homework, and it replies with the review result.
In the workflow version (1-A, workflow), a syntax error gets an error reply first; only code with valid syntax goes on to the AI, which writes feedback containing **"Strengths:" and "Improvements:"**.
If the format check keeps failing, the flow stops when it reaches the maximum number of generations and marks the result as not passed. "Stopped" must never be read as "passed".

**Keep three things apart:** valid syntax (Python can read it) ≠ runs successfully (no error while running) ≠ meets the assignment (matches what the assignment asks for).

We chose it because it happens to use all three kinds of things (and you can also combine them):

| What the helper needs | Example | What we use |
| --- | --- | --- |
| Steps controlled by a flow | Check syntax first; report errors; only on a pass, write feedback and check its format | A **workflow** defined in code (LangGraph, Session 1) |
| Things that need judgment | How to word the feedback, whether to actually run the code and see | Leave it to the **AI** (agent, Session 1) |
| Ways of working and rules | Should the feedback be gentle or strict, what a good variable name looks like | A **handbook** written for the AI to read (Skill, Session 2). Rules that code can check (such as variable naming) can also come with a script or go into the workflow; 2-B compares them |

👉 **This notebook covers:** Run a simplified version of Session 2's division-of-labor diagram once (MCP, workflow, reviewer agent, verifying agent, human confirmation; this version has no assignment attachment and no read_spec), then see where each piece lives once it goes to the cloud.

<details><summary>Open this when you need to look up a component name (no need to memorize it)</summary>

| Component | What it does, in plain words |
| --- | --- |
| `check_syntax` (check syntax) | Only checks whether the code's "grammar" is right, for example whether brackets are closed or a colon is missing. It **does not run the code** |
| `explain_error` (explain the error) | When the syntax is wrong, tells the student in one sentence which line is wrong |
| `give_feedback` (write feedback) | Once a model is connected in 1-A, asks the AI to write "Strengths + Improvements" feedback; the next station checks whether it matches the format |
| `check_format` (check format) | Checks the order of the two markers, that each part is non-empty, and the total character count. It does not verify that the feedback content is correct |
| Tools: `run_code`, `check_indent`, `read_spec` (from 1-B on); `check_naming` (from 2-B on) | Run the homework under teaching limits, check indentation, read the assignment requirements, check variable names |
| Handbooks: `friendly-feedback`, `strict-review`; `naming-check` (2-B), `detailed-feedback` (the 2-C security experiment) | Tell the AI whether feedback should be gentle and encouraging or strict; the naming-check handbook comes with a script; the last one hides a suspicious instruction so you can practice checking an outside handbook |

</details>

**Reminder:** run "0. Setup" first, then follow each part: setup cell → predict → run → compare.
`%%writefile` means "save this cell's content as a file"; edits only take effect after you run the cell. Re-running a setup cell may overwrite the handbooks, so save your own edits first.

## What this notebook does

At the end of Session 2, we imagined upgrading the Homework Helper: fetch 200 homework submissions from GitHub, review them all at once, and send the emails automatically.
We split that into a **division-of-labor diagram**, sorted from two angles: "who decides when it runs" and "how the ability is provided". One feature can use several kinds of components at the same time:

| Role | Plain meaning | Example in this notebook |
| --- | --- | --- |
| Code (workflow) | Order and checks controlled by the workflow | Always check syntax first; always stop and wait for a person before sending email |
| Tool | A Python function the AI can call | `check_naming` (checks naming), `run_code` (runs the homework) |
| MCP | A standard socket for plugging in "tools that live in another program or service" | The homework repo: fetch homework, send email |
| Handbook (Skill) | A work handbook for the AI to read | `friendly-feedback` (friendly feedback) |
| Reviewer agent and verifying agent | The reviewer agent is one configuration run once per submission with that submission's own input (batching, not opening more roles), each on a clean "desk" (its own conversation history); the verifying agent is a second role whose job is separate: it only checks and never writes | The reviewer agent (= the feedback agent on slide 37) uses `friendly-feedback`; the verifying agent is called the "auditor" in the code |

In class we only drew the diagram. This notebook **actually runs it once**, entirely inside free Colab:

| Exercise | What you'll do | Done when | Matching slides |
| --- | --- | --- | --- |
| Exercise 1 | Start a **real MCP server**, the "homework repo", and connect to it through a local socket first | You list the tools and fetch one homework | 48, 52, 39 |
| Exercise 2 ⭐ | **A learning ladder**: review one submission and watch the flow → add human confirmation → run the whole batch | You complete the human-confirmation step before sending | 31, 34-37, 48, 53-54 |
| Exercise 3 | Wrap the flow as an **API service**: start a review by URL, confirm the sending by URL | The API sends and receives requests, and the thread_id is kept across requests | 48, 49, 51, 54 |
| Exercise 4 | **Pre-launch checks**, and generate the deployment files (`Dockerfile`, `langgraph.json`) | Every item on the checklist is checked and the deployment files are generated correctly | 49-50, 55 |

**This notebook does not deploy anything to the cloud** (that needs a cloud account and a credit card). The MCP server and the API service run on this one Colab machine,
and the emails are fake; they are only written to `/content/review-bot/outbox.txt`.

## Deployment map: where each thing lives once it goes to the cloud

| Thing | In this notebook | Where it lives in the cloud | Exercise |
| --- | --- | --- | --- |
| Outer workflow, reviewer agent, verifying agent, local tools (`check_naming`, `run_code`) | Python in Colab | **Your container** (Cloud Run or LangSmith Deployment): a self-built container uses the `server.py` API (`POST /batch`); a managed platform uses its built-in threads / runs API instead | 2, 3 |
| `skills/`, `scripts/` | `/content/review-bot/` | **Shipped with the container**: changing a handbook = releasing a new version | 4 |
| Homework repo, sending email (MCP) | Another program on the same machine | **Someone else's service**: a remote MCP connects through a URL, with authorization set up by that service | 1 |
| Each thread_id's State, and emails waiting for confirmation | A SQLite file | **A database** (Postgres), which survives when the machine is replaced | 2, 3 |
| Model API key | Colab 🔑 Secrets (or none needed) | **A vault** (Secret Manager), exposed as an environment variable | 4 |
| How many to review at once, Trace / Eval | `max_concurrency`, the printed usage | A **task queue**, a **monitoring dashboard** | 2, 4 |

In one sentence: **code, handbooks and scripts travel with the container; data that changes goes into a database; keys go into a vault; other people's abilities are connected over the network.**

The terms in the table: **container** = put a program and everything it needs into one box that runs on any machine; **Cloud Run, LangSmith Deployment** = services that run your program in the cloud;
**Postgres** = a kind of database; **Secret Manager** = a password vault in the cloud; **task queue** = lets jobs wait in line when there is too much work;
**Trace / Eval** = a record of what each step did / automatic tests that give a score.

**When reading the results:** "verified" means the feedback passed the check; it does not mean the student's homework is correct. A homework with a syntax error gets an error notice produced directly by code, which can also be listed as feedback that is ready to send.
2-E always uses the default friendly handbook and verifies the feedback after reviewing; only 2-D demonstrates picking a handbook and a full format redo.

## How to use it

1. Run **"0. Setup"** first (it installs MCP and FastAPI as well, and prepares the tools and handbooks from the earlier notebooks).
2. We suggest doing Exercise 1's stdio first, then Exercise 2 with one submission and the human confirmation; choose the whole batch only after you can explain the result and the confirmation step. Leave HTTP, the API and the deployment files for a second pass.
3. The number of model calls changes with the number of submissions, tool use and rewrites; watch the usage with one submission first. Setup uses fresh default handbooks and does not carry over changes from earlier notebooks.
4. Re-running setup and the `%%writefile` prepare cells overwrites the default files in them, so keep your changes first; the SQLite checkpoint and the fake outbox are not cleared by that. After the environment is rebuilt, prepare the files and services again.
5. This notebook uses `await`: a Colab cell can use `await` directly. Don't worry about it; just press ▶ as usual.

In [ ]:
#@title 0. Setup (run it the first time you open this notebook, or after the runtime is reset)
#@markdown **Model source**: `colab` = the model built into Colab (no API key needed; availability and quota depend on your account); `gemini` = the instructor's Gemini API key (the backup option)
model_source = "colab"  #@param ["colab", "gemini"]
#@markdown If you pick `gemini`: we recommend adding `GOOGLE_API_KEY` under 🔑 "Secrets" in the left sidebar. You can also paste the key below (⚠️ if you paste it, do not share this notebook)
INSTRUCTOR_API_KEY = ""  #@param {type:"string"}
#@markdown Advanced (you usually don't need to change these): set a model name, or leave it empty to use the default
colab_model_name = ""  #@param {type:"string"}
gemini_model_name = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. Install packages with uv (limited to compatible version ranges; time depends on your network)
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ Installing LangGraph / LangChain with uv ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4" "fastapi>=0.115" "uvicorn>=0.30" "mcp>=1.20,<2" "langchain-mcp-adapters>=0.3,<0.4"', shell=True, check=True)

# 2. Write the shared files this notebook uses
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_files = {
    'llm.py': r'''"""Model adapter: wraps either "Colab's built-in model" or "the instructor's Gemini API key" into one model object.

Students don't need to read this file. Every later cell only writes model.invoke(...) and model.bind_tools(...),
and it works the same no matter which provider is behind it.

- colab : google.colab.ai.generate_text (works with a free account, no API key needed)
- gemini: langchain_google_genai.ChatGoogleGenerativeAI (needs GOOGLE_API_KEY)

generate_text takes only one piece of text and returns only one piece of text. It has no native tool calling.
So ColabAIChat adds a small adapter:
1. It joins the messages into one text (system instructions, user, AI, and tool results each get a header)
2. When tools are bound, it appends the tool list and asks the model to "reply with a single line of JSON if you need a tool"
3. It parses the model's JSON into tool_calls, in exactly the same format as native tool calling
This lets you see that the model only ever "says" which tool it wants.
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

SETTINGS_FILE = Path(__file__).with_name(".llm.json")
DEFAULT_GEMINI_MODEL = "gemini-flash-latest"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: about 4 characters per token for English text.
    This is only a guess. A real tokenizer can count noticeably more or fewer."""
    return max(1, round(len(text) / 4))


# ---------- Join the messages into one text ----------

def _describe_tools(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params}): {desc}")
    return "\n".join(lines)


def _messages_to_text(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"=== System instructions ===\n{m.text}")
        elif m.type == "human":
            parts.append(f"=== User ===\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("=== AI ===\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"=== AI ===\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "tool"
            parts.append(f"=== Result of tool {name} ===\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "This time you must call at least one tool.\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"This time you must call {tool_choice}.\n"
        parts.append(
            "=== Tools you can use ===\n"
            f"{_describe_tools(tools)}\n\n"
            "=== Reply rules ===\n"
            f"{must}"
            "If you need to use a tool, reply with a single line of JSON only. Add no other text and no ``` marks:\n"
            '{"tool_calls": [{"name": "tool name", "args": {"parameter name": "parameter value"}}]}\n'
            "If you don't need a tool (for example, you already have enough information), just answer in plain text and output no JSON."
        )
    parts.append("=== AI ===")
    return "\n\n".join(parts)


def _parse_reply(text: str, tools: list[dict] | None) -> AIMessage:
    """If the model replied with a JSON note, turn it into tool_calls. Otherwise treat it as plain text."""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """Wraps google.colab.ai.generate_text as a LangChain chat model."""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _messages_to_text(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _parse_reply(text, tools)
        message.usage_metadata = {
            "input_tokens": estimate_tokens(prompt),
            "output_tokens": estimate_tokens(text),
            "total_tokens": estimate_tokens(prompt) + estimate_tokens(text),
        }
        message.response_metadata["token_count"] = "estimate (Colab's built-in model does not report usage)"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- Public: settings and getting the model ----------

def load_settings() -> dict:
    try:
        return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_settings(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """Remember which model provider was chosen (the API key is not saved)."""
    SETTINGS_FILE.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or DEFAULT_GEMINI_MODEL},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _find_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = load_settings()
    # In the cloud there is no settings file: use the environment variable WORKSHOP_PROVIDER=gemini (the key is read from GOOGLE_API_KEY)
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _find_api_key()
        if not key:
            raise RuntimeError(
                "You chose gemini but no API key was found. Add GOOGLE_API_KEY under 🔑 Secrets on the left, "
                "or paste the key from the instructor into the form in \"0. Environment setup\", then rerun that cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or DEFAULT_GEMINI_MODEL, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai only works on Google Colab. If you are not on Colab, choose gemini instead.") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""Small helpers shared by the workshop notebooks (students don't need to read this).

- prepare()                     : the first cell of every part calls it to reconnect the model and reload edited files
- sample_homework               : the sample homework in the form dropdown
- record() / print_history()    : save the results that experiment cells name, then list them by title in the last cell
- show_graph()                  : draw a LangGraph as a diagram
- save_to_drive() / download_zip()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, estimate_tokens  # noqa: F401  (so notebooks can write: from workshop import estimate_tokens)

__all__ = [
    "BASE", "prepare", "sample_homework", "load_homework", "record", "print_state", "print_order_slip", "print_history", "clear_history",
    "ModelCallCounter", "show_graph", "save_to_drive", "download_zip", "estimate_tokens",
]

BASE = Path(__file__).resolve().parent
HISTORY_FILE = BASE / "history.jsonl"

sample_homework = {
    "Missing parenthesis": "print('hi'\n",
    "Correct program": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "Empty": "",
    "Variable typo": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "Messy indentation": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "Inconsistent naming": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def load_homework(name: str) -> str:
    """When the form says "My homework", read /content/review-bot/my_homework.py. Otherwise take it from the samples."""
    if name.startswith("My homework"):
        path = BASE / "my_homework.py"
        if not path.exists():
            raise FileNotFoundError('There is no "My homework" yet. Run the cell above that has %%writefile ... my_homework.py first.')
        return path.read_text(encoding="utf-8")
    return sample_homework[name]


def prepare():
    """The start of every part: put review-bot on the import path, reload edited files, and return the model."""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _make_mcp_stdio_work_in_notebook()
    return get_model()


def _make_mcp_stdio_work_in_notebook() -> None:
    """mcp's stdio_client connects the MCP server's error output to sys.stderr.
    On Colab (Linux), sys.stderr has a real file descriptor, so this works. Jupyter on Windows has none, so we redirect to a log file."""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State history ----------

def _serializable(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serializable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def record(title: str, content) -> None:
    """Save this result (usually a State) into history.jsonl."""
    row = {"time": _dt.datetime.now().strftime("%H:%M:%S"), "title": title, "content": _serializable(content)}
    with HISTORY_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _short(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"... ({len(text) - n} more characters)"


def print_state(state: dict, indent: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{indent}{k}: ({len(v)} messages)")
            for m in v:
                m = _serializable(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{indent}  [{m['type']}] calls {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{indent}  [{m.get('type')}] {_short(m.get('content', ''), 200)}")
        else:
            print(f"{indent}{k}: {_short(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def print_order_slip(step_no: int, state: dict, prev_state: dict | None = None, skip_keys: tuple = ("code",)) -> None:
    """stream_mode="values" gives the whole order slip at every step. This also marks which fields this step newly wrote."""
    if not prev_state:
        note = "start: blank order slip"
    else:
        changed_keys = [k for k in state if state.get(k) != prev_state.get(k)]
        note = "newly written: " + (", ".join(changed_keys) if changed_keys else "(nothing changed)")
    print(f"Step {step_no} ({note})")
    print("  ", {k: (_short(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in skip_keys})


def print_history() -> None:
    if not HISTORY_FILE.exists():
        print("There are no records yet. Run a few parts above first, then come back to this cell.")
        return
    rows = [json.loads(line) for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 {len(rows)} records (saved in {HISTORY_FILE})\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['time']}] {row['title']} =====")
        content = row["content"]
        if isinstance(content, dict):
            print_state(content)
        elif isinstance(content, list):
            for j, item in enumerate(content, 1):
                print(f"  --- step {j} ---")
                if isinstance(item, dict):
                    print_state(item, "    ")
                else:
                    print(f"    {_short(item)}")
        else:
            print(f"  {_short(content, 1000)}")
        print()


def clear_history() -> None:
    HISTORY_FILE.unlink(missing_ok=True)


# ---------- Count how many times the model was called ----------

class ModelCallCounter(BaseCallbackHandler):
    """Put it in config={"callbacks": [counter]}. It adds 1 every time the model is called."""

    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.count += 1


# ---------- Draw the graph ----------

def show_graph(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("(The drawing service is unreachable for now, so here is a text version. You can paste it into https://mermaid.live to see the diagram.)\n")
        print(g.draw_mermaid())


# ---------- Saving ----------

def save_to_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, destination, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ Saved to Google Drive: {destination}")
    print('   (Save the notebook itself with the menu above: "File → Save a copy in Drive")')


def download_zip() -> None:
    from google.colab import files

    name = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(name), "zip", BASE.parent, BASE.name)
    print(f"✅ Packed: {zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""The Homework Helper's toolbox: run_code, check_indent, read_spec, check_naming."""
''',
    'tools/run_code.py': r'''"""run_code: runs a student's homework under teaching restrictions.

A student's homework could contain anything (import os to delete files, an infinite loop), so this tool has two execution limits plus one readability feature:
1. A check before running: only a few standard modules may be imported (math, random, re, ...). os / sys / subprocess and the like are always blocked.
   eval, exec, open, input and attributes that start with an underscore (often used to get around restrictions) are not allowed either
2. The code runs in a separate child process, and it is stopped by force when the time runs out (so an infinite loop can be stopped too)
3. It returns only one line of result (which line, what error), which is easier for people and models to read. This is not a security limit

⚠️ This is a teaching fence. For a real product, use a real sandbox (a container, RestrictedPython, e2b, Modal).
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

BLOCKED_MODULES = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
ALLOWED_MODULES = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
BLOCKED_FUNCTIONS = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
BLOCKED_ATTRIBUTES = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
TIME_LIMIT = 3
MAX_OUTPUT_CHARS = 800

# The code in the child process that really runs the homework: it uses a "shrunken builtins", and import lets only the allowlist through
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("This tool does not allow import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "homework.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "homework.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _precheck(code: str) -> str | None:
    """A static check before running. If there is a problem, return a one-line explanation. Otherwise return None."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ Line {e.lineno} SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in BLOCKED_MODULES or root not in ALLOWED_MODULES:
                    return f"⛔ Blocked: line {node.lineno} import {name} (this tool does not allow {root})"
        elif isinstance(node, ast.Name) and node.id in BLOCKED_FUNCTIONS:
            return f"⛔ Blocked: line {node.lineno} uses {node.id} (this tool does not allow it)"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in BLOCKED_ATTRIBUTES or node.attr in BLOCKED_MODULES
        ):
            return f"⛔ Blocked: line {node.lineno} accesses .{node.attr} (this tool does not allow it)"
    return None


def run_homework(code: str, time_limit_seconds: float = TIME_LIMIT) -> str:
    """The function that does the real work: check → run in a child process → return one line of result."""
    if not code.strip():
        return "❌ The homework is empty, there is nothing to run"
    blocked = _precheck(code)
    if blocked:
        return blocked
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(ALLOWED_MODULES)), json.dumps(sorted(BLOCKED_FUNCTIONS))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=time_limit_seconds, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ Ran longer than {time_limit_seconds:g} seconds and was stopped by force (there may be an infinite loop)"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ The run failed (the program did not end normally; it may have used too much memory)"
    stdout = result.get("stdout", "")
    if len(stdout) > MAX_OUTPUT_CHARS:
        stdout = stdout[:MAX_OUTPUT_CHARS] + "\n... (output too long, the rest is omitted)"
    if result["ok"]:
        return f"✅ Ran successfully, output:\n{stdout}" if stdout else "✅ Ran successfully (nothing was printed)"
    where = f"line {result['line']} " if result.get("line") else ""
    output = f"\n(printed before the error: {stdout.strip()})" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{output}"


@tool
def run_code(code: str) -> str:
    """Run a piece of Python homework under teaching restrictions and return the output or the error message.

    When to use it: when you want to know what the homework actually prints when it runs, or whether it errors.
    Parameter code: the complete Python code (a string).
    Returns: one line of result, for example "✅ Ran successfully, output: 5" or "❌ line 3 NameError: ...".
    Limits: it can only import a few standard modules such as math, random and re; it cannot use input() or open(); it is stopped after 3 seconds.
    """
    return run_homework(code)
''',
    'tools/check_naming.py': r'''"""check_naming: wraps the Skill's script scripts/check_naming.py as an agent tool.

Wrap the script in @tool and it joins the agent's toolbox.
In this example the AI decides whether to call the tool and tidies up the result. With the same version and the same input, the Python script returns the same check result,
but you still need to confirm whether the model actually called it and whether it kept the result intact.
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

SCRIPT = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def run_naming_script(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(SCRIPT)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    output = proc.stdout.decode("utf-8").strip()
    return output or "The script failed to run: " + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """Check whether variable and function names are lowercase with underscores (snake_case), and return the lines that break the rule.

    When to use it: when reviewing a homework and you need to list the naming problems (this is the script that comes with the naming-check handbook).
    Parameter code: the complete Python code (a string).
    Returns: one line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)". If everything passes it returns "OK: ...".
    """
    return run_naming_script(code)
''',
    'graph.py': r'''"""1-A: the Homework Helper as a workflow. The route is decided by code.

START → check_syntax ─syntax OK→ give_feedback → check_format ─format OK or limit reached→ END
                     └syntax error→ explain_error → END       └failed and retries below the limit→ send back to give_feedback

Three components:
- State: one order slip (a dict) that travels from start to finish
- Node : one workstation (a function). It takes the State and returns only the fields it changed
- Edge : the arrows between workstations. A conditional edge is just an if that returns the name of the next station
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # the homework the student submitted
    syntax_ok: bool | None # is the syntax valid? (None before it is checked)
    error: str             # where the syntax is wrong
    feedback: str          # the feedback for the student
    format_ok: bool | None # does the feedback pass the format check?
    format_problem: str    # why it failed (told to the AI when the feedback is sent back)
    retries: int           # how many times feedback has been generated, including the first


# ---------- Node 1: check_syntax (pure Python): only checks the grammar, never runs the code ----------

_ERROR_TRANSLATIONS = [
    (r"'(.)' was never closed", lambda m: f"missing a closing bracket {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "the string is missing its closing quote"),
    (r"expected ':'", lambda m: "missing a colon :"),
    (r"unexpected indent", lambda m: "extra indentation"),
    (r"expected an indented block", lambda m: "this line needs to be indented"),
    (r"unindent does not match", lambda m: "the indentation doesn't line up"),
    (r"unmatched '(.)'", lambda m: f"an extra {m.group(1)}"),
    (r"invalid character", lambda m: "there is a full-width or unrecognized character"),
    (r"Perhaps you forgot a comma", lambda m: "maybe a comma is missing"),
    (r"invalid syntax", lambda m: "invalid syntax"),
]


def translate_syntax_error(msg: str) -> str:
    for pattern, wording in _ERROR_TRANSLATIONS:
        m = re.search(pattern, msg)
        if m:
            return wording(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "the homework is empty, there is no code"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"line {e.lineno}: {translate_syntax_error(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- Conditional edge 1: syntax OK → write feedback; syntax error → explain the error (just an if) ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2: explain_error (pure Python): one sentence saying which line is wrong ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"Syntax: {state['error']}\nImprovements: fix this line first, then run it again."}


# ---------- Node 3: give_feedback (two versions: fixed text / handed to the AI) ----------

def give_feedback_fixed_text(state: State) -> dict:
    """The version with no AI yet: it returns a fixed piece of text."""
    return {
        "feedback": "Strengths: this homework passes the syntax check.\nImprovements: (this version has no AI yet, so the run result and the logic are not checked)",
        "retries": state.get("retries", 0) + 1,
    }


def make_give_feedback(model, feedback_char_limit: int):
    """The version with AI: the AI is just one line of code living inside this node."""

    def give_feedback(state: State) -> dict:
        prompt = (
            "You are a Python homework helper. Give feedback in English on the homework below.\n"
            "Format rules (you must follow them):\n"
            '- Write only two paragraphs: the first starts with "Strengths:", the second starts with "Improvements:"\n'
            f'- The whole feedback (including "Strengths:", "Improvements:", and line breaks) must be at most {feedback_char_limit} characters\n\n'
            f"Homework:\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\nYour previous feedback was rejected because: {state['format_problem']}. Fix it and rewrite."
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4: check_format (a pure-Python checkpoint): needs two paragraphs and must not be too long ----------

def make_check_format(feedback_char_limit: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        problems = []
        # Check the label order and that both paragraphs are non-empty; line breaks are accepted.
        parts = re.fullmatch(r"\s*Strengths:\s*(.+?)\s*Improvements:\s*(.+?)\s*", feedback, re.DOTALL)
        if not parts or not all(part.strip() for part in parts.groups()):
            problems.append('it needs two paragraphs in order: "Strengths: ..." and "Improvements: ..."')
        if len(feedback) > feedback_char_limit:
            problems.append(f"too long: {len(feedback)} characters (including labels and line breaks), limit {feedback_char_limit} characters")
        return {"format_ok": not problems, "format_problem": "; ".join(problems)}

    return check_format


# ---------- Conditional edge 2: format OK or generation limit reached → end; otherwise send back to rewrite ----------

def make_route_after_format(max_retries: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= max_retries:
            return END  # reached the feedback generation limit; stopping does not mean the format passed
        return "give_feedback"

    return route_after_format


# ---------- Connect the components ----------

def build_helper(model=None, max_retries: int = 3, feedback_char_limit: int = 450):
    """With model=None it uses fixed text. max_retries is a legacy parameter name; it actually counts generations, including the first."""
    give_feedback = make_give_feedback(model, feedback_char_limit) if model else give_feedback_fixed_text

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", make_check_format(feedback_char_limit))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", make_route_after_format(max_retries), ["give_feedback", END])
    return builder.compile()


def blank_order_slip(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}
''',
    'skill_loader.py': r'''"""skill_loader: the AI picks a handbook by looking only at its "card", and the body is opened only after it is picked.

Progressive disclosure has three layers:
1. Card: the name and description between the two --- lines at the top of SKILL.md. They are given to the model when it picks
2. Body: the whole SKILL.md, read only after it is picked
3. Appendix: files in scripts/ and references/, opened only when needed
"""

from pathlib import Path

SKILLS = Path(__file__).resolve().parent / "skills"


def read_card(path: Path) -> dict:
    """Read only the name and description between the two --- lines."""
    text = Path(path).read_text(encoding="utf-8")
    card = {"name": Path(path).parent.name, "description": "", "path": str(path)}
    if text.startswith("---"):
        head = text.split("---", 2)[1]
        for line in head.strip().splitlines():
            key, _, value = line.partition(":")
            if key.strip() in ("name", "description"):
                card[key.strip()] = value.strip()
    return card


def read_body(card: dict) -> str:
    """Opened only after the handbook is picked: the handbook content below the card."""
    text = Path(card["path"]).read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text


def list_cards(only: list[str] | None = None) -> list[dict]:
    cards = [read_card(p) for p in sorted(SKILLS.glob("*/SKILL.md"))]
    if only:
        cards = [c for c in cards if Path(c["path"]).parent.name in only]
    return cards


def pick_handbook(model, request: str, cards: list[dict]) -> dict:
    """Show the model all the cards and let it pick one. Returns the card that was picked."""
    if not cards:
        raise ValueError("No handbook is available. Write a SKILL.md first, then list the cards.")
    card_list = "\n".join(f"- {c['name']}: {c['description']}" for c in cards)
    answer = model.invoke(
        "Pick the one handbook that best fits the review request below.\n"
        f"Review request: {request}\n\n"
        f"Handbook cards:\n{card_list}\n\n"
        "Answer with only the handbook's name, no other words."
    ).text.strip()
    name = answer.strip("`\"' \n")
    options = {c["name"]: c for c in cards}
    if name not in options:
        raise ValueError(f"The model did not return exactly one valid handbook name: {answer!r}. Valid names: {list(options)}. Check the cards and the model output; no handbook was picked this time.")
    return options[name]
''',
    'spec.md': r'''Write a function add(a, b) that returns the sum of two numbers, and use print(add(2, 3)) to print 5.
''',
    'deploy/__init__.py': r'''"""2-E: getting ready for the cloud (API service, parallel review, human confirmation before sending, pre-launch checks)."""
''',
    'deploy/homework_mcp.py': r'''"""Homework repo MCP server: the tools live in "another program" and plug in through the MCP socket.

It's just an ordinary Python file; the functions simply have @mcp.tool() on top.
In the real world this role is played by GitHub's MCP server (fetching homework) and a mailbox MCP server (sending email);
here we use fake data instead, and "sending email" only writes to outbox.txt.

Two ways to connect:
    python homework_mcp.py              # stdio: a local socket; the agent side starts it
    python homework_mcp.py --http 8765  # Streamable HTTP: it becomes a URL, like a remote MCP
"""

import argparse
import datetime as _dt
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

OUTBOX = Path(__file__).resolve().parent.parent / "outbox.txt"

# A fake homework repo: students, emails, submitted homework
SUBMISSIONS = {
    "Alex": ("alex@example.com", "print('hi'\n"),
    "Ben": ("ben@example.com", "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n"),
    "Chloe": ("chloe@example.com", "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n"),
    "Jay": ("jay@example.com", "def CalcAverage(scoreList):\n    Total = 0\n    for s in scoreList:\n        Total += s\n"
                               "    return Total / len(scoreList)\n\nmyScores = [90, 85, 77]\nprint(CalcAverage(myScores))\n"),
    "Fiona": ("fiona@example.com", "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n"
                                   "  return total / len(scores)\n\nprint(average([90, 85, 77]))\n"),
}

mcp = FastMCP("homework_repo", log_level="WARNING")


@mcp.tool()
def list_homework() -> str:
    """List the students who have submitted homework (name and email) as a JSON string. Call this first when you need to review homework."""
    return json.dumps([{"student": name, "email": email} for name, (email, _) in SUBMISSIONS.items()], ensure_ascii=False)


@mcp.tool()
def get_homework(student: str) -> str:
    """Get the homework code one student submitted. Parameter student: the student's name (from list_homework)."""
    if student not in SUBMISSIONS:
        return f'No homework found for "{student}"'
    return SUBMISSIONS[student][1]


@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a student (irreversible: once sent, it can't be taken back). Parameters: recipient email, subject, body."""
    with OUTBOX.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] To {to} | {subject}\n{body}\n\n")
    return f"(simulated) Sent to {to}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--http", type=int, help="Serve over Streamable HTTP on this port (omit it to use stdio)")
    args = parser.parse_args()
    if args.http:
        mcp.settings.host = "127.0.0.1"
        mcp.settings.port = args.http
        mcp.run(transport="streamable-http")   # URL: http://127.0.0.1:<port>/mcp
    else:
        mcp.run()
''',
    'deploy/multi_agent.py': r'''"""2-E Exercise 2: run Session 2's whole division-of-labor diagram once in Colab (multi-agent + MCP).

START → fetch_homework ─Send×N→ review_one → confirm_before_send ─send→ send_emails → END
          (MCP)                     │            (interrupt)      └Send none→ END       (MCP)
                                    └ code checks syntax → reviewer agent (handbook + script + run_code) → verification agent

How this maps to the five roles you can combine in the diagram (this is a simplified version: no assignment attachment and no read_spec):
- Code: the order of the outer flow, the syntax check, "if verification fails, rewrite once", and ending right away on "Send none"
- Tool: check_naming (the script that comes with the handbook), run_code
- MCP: the homework repo's list_homework / get_homework / send_email (the tools live in another program)
- Skill: the friendly-feedback handbook, used as the reviewer agent's system prompt
- Agent: the same reviewer agent handles each submission (dispatched with Send, each on its own clean desk) + a verification agent with its own separate job
"""

import json
import operator
import re
import sys
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send, interrupt

from graph import check_syntax
from skill_loader import list_cards, read_body
from tools.check_naming import check_naming
from tools.run_code import run_code

MCP_SERVER = Path(__file__).resolve().with_name("homework_mcp.py")
CHECKPOINT_PATH = Path(__file__).resolve().with_name("multi_agent.sqlite")
SEND_OPTIONS = ["Send all", "Send only verified", "Send none"]


# ---------- MCP: the homework repo's socket ----------

def homework_repo_config(url: str | None = None) -> dict:
    """Given a URL, connect to the remote server (Streamable HTTP); without one, start a server locally (stdio)."""
    if url:
        return {"homework_repo": {"url": url, "transport": "streamable_http"}}
    return {"homework_repo": {"command": sys.executable, "args": [str(MCP_SERVER)], "transport": "stdio"}}


async def get_mcp_tools(config: dict) -> dict:
    return {t.name: t for t in await MultiServerMCPClient(config).get_tools()}


def result_text(result) -> str:
    """An MCP tool returns a list of content blocks; pull out the text inside them."""
    if isinstance(result, str):
        return result
    if isinstance(result, list):
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in result)
    return str(result)


# ---------- State ----------

class BatchState(TypedDict, total=False):
    roster: list[dict]                            # [{"student", "email", "code"}]
    results: Annotated[list[dict], operator.add]  # the result each "review_one" hands back, appended in order
    decision: str
    sent_log: list[str]


class SubmissionState(TypedDict):
    student: str
    email: str
    code: str


# ---------- Verification agent: checks the homework and the feedback independently; it does not write feedback ----------

AUDITOR_PERSONA = (
    "You are a strict auditor. Your only job is to check homework feedback that someone else wrote; you never write feedback yourself.\n"
    "Check three things: 1. Do the problems the feedback mentions really exist in the homework? "
    "2. Does it cite any line numbers that are wrong? 3. Does it use harsh words such as \"wrong\" or \"terrible\"?\n"
    'Reply with exactly one line of JSON and nothing else: {"passed": true or false, "reason": "one sentence"}'
)


def verify(model, code: str, feedback: str) -> dict:
    # What code can decide, check with code first (these checks cost no tokens); the auditor still calls the model once per submission
    problems = []
    if "Strengths" not in feedback or "Improvements" not in feedback:
        problems.append('Missing the "Strengths" or "Improvements" section')
    line_count = len(code.splitlines())
    for n in re.findall(r"(?i)\bline\s*(\d+)", feedback):
        if not 1 <= int(n) <= line_count:
            problems.append(f"Mentions line {n}, but the homework has only {line_count} lines")
    # What code can't decide goes to the auditor
    with_line_numbers = "\n".join(f"{i:>2} | {line}" for i, line in enumerate(code.splitlines(), 1))
    answer = model.invoke([SystemMessage(AUDITOR_PERSONA),
                           HumanMessage(f"Homework:\n{with_line_numbers}\n\nFeedback:\n{feedback}")]).text
    m = re.search(r"\{.*\}", answer, re.S)
    try:
        judgement = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        judgement = {}
    if not isinstance(judgement, dict) or not isinstance(judgement.get("passed"), bool):
        problems.append("Could not parse the auditor's reply into a true/false passed value; check it by hand")
    elif not judgement["passed"]:
        problems.append("Auditor: " + str(judgement.get("reason", answer.strip()[:60])))
    return {"passed": not problems, "reason": "; ".join(problems) or "OK"}


# ---------- Wiring the components together ----------

def build_multi_agent_helper(model, mcp_config: dict | None = None, checkpointer=None, only_students: str | None = None):
    mcp_config = mcp_config or homework_repo_config()
    handbook = read_body(list_cards(["friendly-feedback"])[0])
    REVIEWER_PERSONA = (
        f"{handbook}\n\n"
        "# Your tools\n"
        "- check_naming: checks variable naming (the script that comes with the handbook); use it when you discuss naming\n"
        "- run_code: runs the homework so you can see its output\n"
        "You decide whether to use them and which one. Keep the feedback to at most 750 characters."
    )

    async def fetch_homework(state: BatchState) -> dict:
        # Fetching homework needs no judgment, so "code" calls the MCP tools, not an agent
        tools = await get_mcp_tools(mcp_config)
        roster = json.loads(result_text(await tools["list_homework"].ainvoke({})))
        if only_students is not None:
            roster = [a for a in roster if a["student"] == only_students]
            if not roster:
                raise ValueError(f"The homework repo has no student named: {only_students}")
        for a in roster:
            a["code"] = result_text(await tools["get_homework"].ainvoke({"student": a["student"]}))
        return {"roster": roster}

    def dispatch(state: BatchState) -> list[Send]:
        # Start one "review_one" per submission: each gets its own clean desk
        return [Send("review_one", a) for a in state["roster"]]

    def review_one(state: SubmissionState) -> dict:
        code = state["code"]
        syntax = check_syntax({"code": code})
        if not syntax["syntax_ok"]:   # Anything you can write as an if: the code replies directly, no AI call
            feedback = f"Syntax: {syntax['error']}\nImprovements: Fix this line first, then run the program again."
            return {"results": [{**state, "feedback": feedback, "used_tools": [], "rewrite_count": 0,
                                 "verify": {"passed": True, "reason": "The code replied to the syntax error directly; no AI was involved"}}]}
        reviewer = create_agent(model, tools=[check_naming, run_code], system_prompt=REVIEWER_PERSONA)
        messages = [{"role": "user", "content": f"Please review this homework:\n```python\n{code}\n```"}]
        used_tools, rewrite_count = [], 0
        while True:
            result = reviewer.invoke({"messages": messages}, config={"recursion_limit": 25})
            # result includes the full record of the previous round; take only this round's new messages so old tool requests aren't counted twice.
            used_tools += [c["name"] for m in result["messages"][len(messages):] for c in (getattr(m, "tool_calls", None) or [])]
            feedback = result["messages"][-1].text.strip()
            verdict = verify(model, code, feedback)
            if verdict["passed"] or rewrite_count >= 1:      # If verification fails, send it back for at most one rewrite
                break
            rewrite_count += 1
            messages = result["messages"] + [{"role": "user", "content": f"The auditor sent it back: {verdict['reason']}. Fix it and rewrite the feedback."}]
        return {"results": [{**state, "feedback": feedback, "used_tools": used_tools,
                             "rewrite_count": rewrite_count, "verify": verdict}]}

    def confirm_before_send(state: BatchState) -> dict:
        # Sending email is irreversible: stop and hand what would be sent to a person to read
        emails = [{"student": r["student"], "email": r["email"], "verified": r["verify"]["passed"],
                   "feedback": r["feedback"]} for r in state["results"]]
        decision = interrupt({"emails_to_send": emails, "options": SEND_OPTIONS})
        if decision == "Send none":
            return {"decision": decision, "sent_log": ["No emails were sent (decision: Send none)"]}
        return {"decision": decision}

    def should_send(state: BatchState) -> str:
        # Choosing "Send none" ends right away, without entering the send_emails node
        return END if state.get("decision") == "Send none" else "send_emails"

    async def send_emails(state: BatchState) -> dict:
        to_send = [r for r in state["results"]
                   if state.get("decision") == "Send all" or (state.get("decision") == "Send only verified" and r["verify"]["passed"])]
        if not to_send:
            return {"sent_log": [f"No emails were sent (decision: {state.get('decision')})"]}
        tools = await get_mcp_tools(mcp_config)
        log = []
        for r in to_send:
            result = await tools["send_email"].ainvoke({"to": r["email"], "subject": "Homework feedback", "body": r["feedback"]})
            log.append(result_text(result))
        return {"sent_log": log}

    builder = StateGraph(BatchState)
    builder.add_node("fetch_homework", fetch_homework)
    builder.add_node("review_one", review_one)
    builder.add_node("confirm_before_send", confirm_before_send)
    builder.add_node("send_emails", send_emails)
    builder.add_edge(START, "fetch_homework")
    builder.add_conditional_edges("fetch_homework", dispatch, ["review_one"])
    builder.add_edge("review_one", "confirm_before_send")
    builder.add_conditional_edges("confirm_before_send", should_send, ["send_emails", END])
    builder.add_edge("send_emails", END)
    return builder.compile(checkpointer=checkpointer)


async def open_checkpointer(path: Path = CHECKPOINT_PATH):
    """interrupt waits for a person to confirm, so the State must be saved; here it goes into a SQLite file (in the cloud, use Postgres)."""
    import aiosqlite
    from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

    return AsyncSqliteSaver(await aiosqlite.connect(str(path)))


class usage_meter(BaseCallbackHandler):
    """Counts how many times the model is called and adds up the tokens (for Colab's built-in model, the tokens are estimates)."""

    def __init__(self):
        self.count = 0
        self.input_tokens = 0
        self.output_tokens = 0

    def on_llm_end(self, response, **kwargs):
        self.count += 1
        for generations in response.generations:
            for g in generations:
                usage = getattr(getattr(g, "message", None), "usage_metadata", None) or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)
''',
    'skills/friendly-feedback/SKILL.md': r'''---
name: friendly-feedback
description: Gentle, encouraging homework feedback for Python beginners. Starts with strengths, then suggests improvements. Good for first-time submissions and students who just learned Python.
---

# Friendly Feedback Rules

1. Start with one thing the student did well
2. Point out at most two problems, and give a fix for each
3. Don't use words like "wrong" or "terrible"
4. Write the feedback in two paragraphs: the first starts with "Strengths:", the second starts with "Improvements:"
''',
    'skills/strict-review/SKILL.md': r'''---
name: strict-review
description: Strict, thorough review for midterms and formal grading. Lists every problem, no softening, and includes point deductions.
---

# Strict Review Rules

1. List every problem, miss none, and give the line number for each
2. No pleasantries, and no words like "great" or "nice"
3. Give the points deducted for each problem (100 points in total)
4. Write the feedback in two paragraphs: the first starts with "Strengths:" (if there really are none, write "Strengths: none"), the second starts with "Improvements:"
''',
    'skills/naming-check/SKILL.md': r'''---
name: naming-check
description: Checks whether the variable and function names in a Python homework are lowercase English with underscores (snake_case). Use it when reviewing a homework and you need to list the naming problems.
---

# Naming Check

Run scripts/check_naming.py (the tool is named check_naming) and write its output into the feedback.
Don't judge by yourself, and don't add or drop any item. Copy every line it prints.

Shared rules: check assigned variables, function names and function parameters; for repeated names, list only the first definition; don't check class names; skip self and cls.
A name may have one leading underscore, followed by a lowercase English letter, and then lowercase English letters, digits or underscores.
This exercise has no exception for constants: MAX fails; result_2 passes. The script runs these rules.

Output format: copy every line the script prints, for example "Line 3: myVar breaks the rule (suggest my_var)"
''',
    'skills/naming-check/scripts/check_naming.py': r'''"""Check whether variable and function names are lowercase English with underscores (snake_case).

Usage:
    python check_naming.py homework.py        # check one file
    python check_naming.py < homework.py      # or read from standard input
Output:
    One line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)"
    When everything passes it prints "OK: all variable and function names are lowercase with underscores"

A script does one thing and prints only a few short lines, so people and the AI can both check it. The same input and the same script give the same result.
"""

import ast
import re
import sys

RULE = re.compile(r"^_?[a-z][a-z0-9_]*$")


def suggest_name(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def find_all_names(tree: ast.AST) -> dict[str, int]:
    """name → line number of its first appearance (variables, functions, parameters; class names are not included)."""
    names = {}

    def note(name, line):
        if name not in names or line < names[name]:
            names[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            note(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            note(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            note(node.arg, node.lineno)
    return names


def check_names(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"Cannot check: line {e.lineno} has a syntax error"
    problems = [
        f"Line {line}: {name} breaks the rule (suggest {suggest_name(name)})"
        for name, line in sorted(find_all_names(tree).items(), key=lambda x: (x[1], x[0]))
        if not RULE.match(name)
    ]
    return "\n".join(problems) if problems else "OK: all variable and function names are lowercase with underscores"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(check_names(source))
''',
}
for rel, content in _files.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# Make Python look again for the files we just wrote (if another cell ran first, Python remembers "this folder does not exist")
import importlib
importlib.invalidate_caches()

# 3. Remember the model source (the API key is not saved) and test the connection
import llm
llm.save_settings(model_source, colab_model_name.strip(), gemini_model_name.strip())
if INSTRUCTOR_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = INSTRUCTOR_API_KEY.strip()
from workshop import *
model = prepare()

reply = model.invoke("Say hello in one sentence to a classmate who just learned Python").text
print(f"✅ Setup complete. Model source: {model_source}")
print(f"🤖 The model says: {reply.strip()}")

---
## Exercise 1: Start a real MCP server

> 📍 **Matching slides**: slide 48 (deployment map), 52 (remote MCP); Session 1 slide 39 (MCP sockets)

First build the "someone else's service" box from the diagram. An MCP tool lives in **another program**. The next cell writes `deploy/homework_mcp.py`: it is an ordinary Python file,
and the `@mcp.tool()` on top of three functions turns them into tools that can be plugged in through the socket:

This exercise's data source is a Python dictionary in `homework_mcp.py`, with fake mailboxes and homework code for **5 students** already filled in; it does not read any student repo on GitHub yet.
The MCP server and the connection are real, but the data source and the email sending are simulated.

| Tool | What this exercise does | When connecting a real service |
| --- | --- | --- |
| `list_homework`, `get_homework` | Take the roster and code from the default dictionary | Connect to the GitHub API, or adapt a compatible MCP tool |
| `send_email` | Only writes into a simulated outbox | Connect a real email service |

These are tool names defined for this course; you can't just swap the URL for any GitHub MCP server and expect it to work. The tool names, parameters and return format must match too.

Two ways to connect: **stdio** is a local socket (the agent side starts the server as a child process); **Streamable HTTP** is a URL.
That is how a remote MCP works, except that a real service may use **OAuth** sign-in or another kind of authorization (the standard way of signing in with an account and exchanging it for a pass).

### 🤔 Prediction questions
1. Once connected, how does an MCP tool look different from the `@tool` you wrote yourself in 1-B?
2. When you connect by URL instead, will the tool names and usage change?

In [ ]:
%%writefile /content/review-bot/deploy/homework_mcp.py
"""Homework repo MCP server: the tools live in "another program" and plug in through the MCP socket.

It's just an ordinary Python file; the functions simply have @mcp.tool() on top.
In the real world this role is played by GitHub's MCP server (fetching homework) and a mailbox MCP server (sending email);
here we use fake data instead, and "sending email" only writes to outbox.txt.

Two ways to connect:
    python homework_mcp.py              # stdio: a local socket; the agent side starts it
    python homework_mcp.py --http 8765  # Streamable HTTP: it becomes a URL, like a remote MCP
"""

import argparse
import datetime as _dt
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

OUTBOX = Path(__file__).resolve().parent.parent / "outbox.txt"

# A fake homework repo: students, emails, submitted homework
SUBMISSIONS = {
    "Alex": ("alex@example.com", "print('hi'\n"),
    "Ben": ("ben@example.com", "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n"),
    "Chloe": ("chloe@example.com", "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n"),
    "Jay": ("jay@example.com", "def CalcAverage(scoreList):\n    Total = 0\n    for s in scoreList:\n        Total += s\n"
                               "    return Total / len(scoreList)\n\nmyScores = [90, 85, 77]\nprint(CalcAverage(myScores))\n"),
    "Fiona": ("fiona@example.com", "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n"
                                   "  return total / len(scores)\n\nprint(average([90, 85, 77]))\n"),
}

mcp = FastMCP("homework_repo", log_level="WARNING")


@mcp.tool()
def list_homework() -> str:
    """List the students who have submitted homework (name and email) as a JSON string. Call this first when you need to review homework."""
    return json.dumps([{"student": name, "email": email} for name, (email, _) in SUBMISSIONS.items()], ensure_ascii=False)


@mcp.tool()
def get_homework(student: str) -> str:
    """Get the homework code one student submitted. Parameter student: the student's name (from list_homework)."""
    if student not in SUBMISSIONS:
        return f'No homework found for "{student}"'
    return SUBMISSIONS[student][1]


@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a student (irreversible: once sent, it can't be taken back). Parameters: recipient email, subject, body."""
    with OUTBOX.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] To {to} | {subject}\n{body}\n\n")
    return f"(simulated) Sent to {to}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--http", type=int, help="Serve over Streamable HTTP on this port (omit it to use stdio)")
    args = parser.parse_args()
    if args.http:
        mcp.settings.host = "127.0.0.1"
        mcp.settings.port = args.http
        mcp.run(transport="streamable-http")   # URL: http://127.0.0.1:<port>/mcp
    else:
        mcp.run()


In [ ]:
#@title ▶ Run: connect to the homework repo through a local socket (stdio)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json
from langchain_core.utils.function_calling import convert_to_openai_tool
from deploy import multi_agent

config = multi_agent.homework_repo_config()          # stdio: this side starts homework_mcp.py as another program
mcp_tools = await multi_agent.get_mcp_tools(config)
for t in mcp_tools.values():
    print(f"🔌 {t.name} (parameters: {list(t.args) or 'none'}): {t.description.splitlines()[0]}")

roster = json.loads(multi_agent.result_text(await mcp_tools["list_homework"].ainvoke({})))
print("\nlist_homework →", ", ".join(a["student"] for a in roster))
print("\nget_homework(student='Chloe') →")
print(multi_agent.result_text(await mcp_tools["get_homework"].ainvoke({"student": "Chloe"})))

tool_spec_tokens = estimate_tokens(json.dumps([convert_to_openai_tool(t) for t in mcp_tools.values()], ensure_ascii=False))
print(f"The tool descriptions of these {len(mcp_tools)} MCP tools add up to about {tool_spec_tokens} tokens (estimate)")
record("2-E Exercise 1: connected to the homework repo over stdio", {"tools": list(mcp_tools), "students": [a["student"] for a in roster], "tool_spec_tokens": tool_spec_tokens})

In [ ]:
#@title ▶ Optional: turn the same server into a URL (Streamable HTTP, like a remote MCP)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import asyncio, socket, subprocess
from deploy import multi_agent

old_process = getattr(sys, "_homework_repo_process", None)       # on a re-run, shut down the one started last time
if old_process:
    old_process.terminate()
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
log_file = open(BASE / "deploy" / "homework_mcp.log", "w")
sys._homework_repo_process = subprocess.Popen([sys.executable, str(multi_agent.MCP_SERVER), "--http", str(port)],
                                              stdout=log_file, stderr=log_file)
url = f"http://127.0.0.1:{port}/mcp"
for _ in range(60):
    try:
        mcp_tools = await multi_agent.get_mcp_tools(multi_agent.homework_repo_config(url))
        break
    except Exception:
        await asyncio.sleep(0.5)
else:
    raise SystemExit("⚠️ The MCP server did not start. Check /content/review-bot/deploy/homework_mcp.log")

(BASE / "deploy" / ".mcp_port").write_text(str(port))
print(f"🌐 The homework repo MCP is now a URL: {url}")
print("Tools connected by URL:", ", ".join(mcp_tools))
print("\nget_homework(student='Jay') →")
print(multi_agent.result_text(await mcp_tools["get_homework"].ainvoke({"student": "Jay"})))
record("2-E Exercise 1: connected to the homework repo by URL", {"url": url, "tools": list(mcp_tools)})

### 👀 What did you see?

- An MCP tool also has a name, a description and parameters. **Once connected as a tool, the model still proposes calls from the name, description and parameters**; at run time there is also a connection to another program and error handling to think about.
  (This Python adapter example waits for the tool's return with `await ...ainvoke()`; that is not how every MCP client is written.)
- Connected by URL, the tools are exactly the same. In the cloud, the homework repo is just a URL on someone else's service: your program doesn't need to know how it was written or in what language.
- MCP provides a list of tools; the application decides which tools to offer the model. A Skill can describe how to use them, but it does not shrink the tool definitions or revoke permissions automatically. In this flow the code calls MCP directly, and the reviewer agent gets only the naming-check and run-code tools.
- **Done when**: you find the tool list in the stdio output (at least `list_homework` and `get_homework`), and you can fetch Chloe's homework. Once you can explain "another program provides the tools", try the URL connection as an option; you don't need to understand network deployment first.

---
## Exercise 2 ⭐: The review flow, from one submission to the whole batch

> 📍 **Matching slides**: slide 31 "Feature 1: fetching homework from GitHub", 34 "Reviewing 200 submissions: independent first, then parallel", 35 "Verifying feedback: rules first, then independent judgment", 36 "Before sending email, make the flow wait for confirmation", 37 "The Homework Helper's full division of labor", 48 "The division of labor in the cloud: where each box lives", 53 "Reviewing 200 at once: Send", 54 "Waiting for confirmation before sending: interrupt"

Exercise 1 started the homework repo. This exercise uses it to run the simplified diagram once, in **three stages** (one submission goes all the way through → add human confirmation → run the whole batch). The next cell writes `deploy/multi_agent.py`; this version has no assignment attachment and no `read_spec`:

| Box in the diagram | Role | In the code |
| --- | --- | --- |
| Fetch homework from the homework repo | MCP (called by the **code**, no judgment needed) | The `fetch_homework` node calls `list_homework` and `get_homework` |
| Always check syntax first | Code | On a syntax error it replies directly and **does not call the AI** |
| Review each submission once | The same reviewer agent (= the feedback agent on slide 37) + Skill + script: batching, not opening more roles | `Send` dispatches each submission to `review_one`; the handbook is `friendly-feedback`, and the tools are `check_naming` and `run_code` |
| Check the feedback for made-up claims | The verifying agent (a separate checking job, called the "auditor" in the code) + code | Code checks the line numbers first, then the auditor reads the content; a failure is sent back for at most one rewrite; if it still fails, the failed state is kept. The auditor gets only the original homework and the feedback, with **no assignment requirements**, so it cannot judge whether the assignment was misunderstood (slide 35's full design needs the assignment + homework + draft) |
| Send the email | MCP + human confirmation | `interrupt()` stops, and only after a person decides is `send_email` called; choosing "Send none" ends the run right there |

Two new pieces of syntax in the table:
- `Send`: sends each item of a list to the `review_one` node, so they can be handled at the same time. The node checks syntax first and creates a reviewer agent only if it passes.
- `interrupt()`: when the flow gets here it stops, saves the State, and waits for a person to answer before it continues.

The flow chart has only four stations (fetch_homework → review_one → confirm_before_send → send_emails); the syntax check, the review and the verification are all written inside the `review_one` node, run once per submission (slide 37 draws them as separate boxes). When you choose "Send none", the flow ends straight from `confirm_before_send` without entering `send_emails`.

**The learning ladder: three stages, growing step by step**

| Stage | Form setting | What you'll see | Done when |
| --- | --- | --- | --- |
| Step 1 | Scope "Just Chloe", concurrency 1 | One full review flow, the verification result, and the emails waiting to be sent | You see "⏸️ Stopped before sending" |
| Step 2 | Decide on sending ("Send only verified" / another option) | After the human confirmation, the log of what was sent | You see the "sent_log" and the outbox content |
| Advanced | Change scope to "All five", concurrency to 3 | Five submissions reviewed in parallel, with time and resource comparison | Try it after finishing the second round (all five, concurrency 1) |

Note "don't pass the work along three people": one submission, from review to fix, is handled by the **same reviewer**; the semantic judgment goes to a separate agent whose only job is to check; independent checking can still miss errors.

### 🤔 Prediction questions
1. First round, Chloe only: predict which node will stop and how you can tell that nothing has been sent yet. Second round, all five: Alex has a syntax error, so how many submissions need the model to review them?
2. If you change "max_concurrency" from 1 to 3, will it be faster? Will it use fewer tokens?
3. When Step 1 finishes, has the email been sent?

In [ ]:
%%writefile /content/review-bot/deploy/multi_agent.py
"""2-E Exercise 2: run Session 2's whole division-of-labor diagram once in Colab (multi-agent + MCP).

START → fetch_homework ─Send×N→ review_one → confirm_before_send ─send→ send_emails → END
          (MCP)                     │            (interrupt)      └Send none→ END       (MCP)
                                    └ code checks syntax → reviewer agent (handbook + script + run_code) → verification agent

How this maps to the five roles you can combine in the diagram (this is a simplified version: no assignment attachment and no read_spec):
- Code: the order of the outer flow, the syntax check, "if verification fails, rewrite once", and ending right away on "Send none"
- Tool: check_naming (the script that comes with the handbook), run_code
- MCP: the homework repo's list_homework / get_homework / send_email (the tools live in another program)
- Skill: the friendly-feedback handbook, used as the reviewer agent's system prompt
- Agent: the same reviewer agent handles each submission (dispatched with Send, each on its own clean desk) + a verification agent with its own separate job
"""

import json
import operator
import re
import sys
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send, interrupt

from graph import check_syntax
from skill_loader import list_cards, read_body
from tools.check_naming import check_naming
from tools.run_code import run_code

MCP_SERVER = Path(__file__).resolve().with_name("homework_mcp.py")
CHECKPOINT_PATH = Path(__file__).resolve().with_name("multi_agent.sqlite")
SEND_OPTIONS = ["Send all", "Send only verified", "Send none"]


# ---------- MCP: the homework repo's socket ----------

def homework_repo_config(url: str | None = None) -> dict:
    """Given a URL, connect to the remote server (Streamable HTTP); without one, start a server locally (stdio)."""
    if url:
        return {"homework_repo": {"url": url, "transport": "streamable_http"}}
    return {"homework_repo": {"command": sys.executable, "args": [str(MCP_SERVER)], "transport": "stdio"}}


async def get_mcp_tools(config: dict) -> dict:
    return {t.name: t for t in await MultiServerMCPClient(config).get_tools()}


def result_text(result) -> str:
    """An MCP tool returns a list of content blocks; pull out the text inside them."""
    if isinstance(result, str):
        return result
    if isinstance(result, list):
        return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in result)
    return str(result)


# ---------- State ----------

class BatchState(TypedDict, total=False):
    roster: list[dict]                            # [{"student", "email", "code"}]
    results: Annotated[list[dict], operator.add]  # the result each "review_one" hands back, appended in order
    decision: str
    sent_log: list[str]


class SubmissionState(TypedDict):
    student: str
    email: str
    code: str


# ---------- Verification agent: checks the homework and the feedback independently; it does not write feedback ----------

AUDITOR_PERSONA = (
    "You are a strict auditor. Your only job is to check homework feedback that someone else wrote; you never write feedback yourself.\n"
    "Check three things: 1. Do the problems the feedback mentions really exist in the homework? "
    "2. Does it cite any line numbers that are wrong? 3. Does it use harsh words such as \"wrong\" or \"terrible\"?\n"
    'Reply with exactly one line of JSON and nothing else: {"passed": true or false, "reason": "one sentence"}'
)


def verify(model, code: str, feedback: str) -> dict:
    # What code can decide, check with code first (these checks cost no tokens); the auditor still calls the model once per submission
    problems = []
    if "Strengths" not in feedback or "Improvements" not in feedback:
        problems.append('Missing the "Strengths" or "Improvements" section')
    line_count = len(code.splitlines())
    for n in re.findall(r"(?i)\bline\s*(\d+)", feedback):
        if not 1 <= int(n) <= line_count:
            problems.append(f"Mentions line {n}, but the homework has only {line_count} lines")
    # What code can't decide goes to the auditor
    with_line_numbers = "\n".join(f"{i:>2} | {line}" for i, line in enumerate(code.splitlines(), 1))
    answer = model.invoke([SystemMessage(AUDITOR_PERSONA),
                           HumanMessage(f"Homework:\n{with_line_numbers}\n\nFeedback:\n{feedback}")]).text
    m = re.search(r"\{.*\}", answer, re.S)
    try:
        judgement = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        judgement = {}
    if not isinstance(judgement, dict) or not isinstance(judgement.get("passed"), bool):
        problems.append("Could not parse the auditor's reply into a true/false passed value; check it by hand")
    elif not judgement["passed"]:
        problems.append("Auditor: " + str(judgement.get("reason", answer.strip()[:60])))
    return {"passed": not problems, "reason": "; ".join(problems) or "OK"}


# ---------- Wiring the components together ----------

def build_multi_agent_helper(model, mcp_config: dict | None = None, checkpointer=None, only_students: str | None = None):
    mcp_config = mcp_config or homework_repo_config()
    handbook = read_body(list_cards(["friendly-feedback"])[0])
    REVIEWER_PERSONA = (
        f"{handbook}\n\n"
        "# Your tools\n"
        "- check_naming: checks variable naming (the script that comes with the handbook); use it when you discuss naming\n"
        "- run_code: runs the homework so you can see its output\n"
        "You decide whether to use them and which one. Keep the feedback to at most 750 characters."
    )

    async def fetch_homework(state: BatchState) -> dict:
        # Fetching homework needs no judgment, so "code" calls the MCP tools, not an agent
        tools = await get_mcp_tools(mcp_config)
        roster = json.loads(result_text(await tools["list_homework"].ainvoke({})))
        if only_students is not None:
            roster = [a for a in roster if a["student"] == only_students]
            if not roster:
                raise ValueError(f"The homework repo has no student named: {only_students}")
        for a in roster:
            a["code"] = result_text(await tools["get_homework"].ainvoke({"student": a["student"]}))
        return {"roster": roster}

    def dispatch(state: BatchState) -> list[Send]:
        # Start one "review_one" per submission: each gets its own clean desk
        return [Send("review_one", a) for a in state["roster"]]

    def review_one(state: SubmissionState) -> dict:
        code = state["code"]
        syntax = check_syntax({"code": code})
        if not syntax["syntax_ok"]:   # Anything you can write as an if: the code replies directly, no AI call
            feedback = f"Syntax: {syntax['error']}\nImprovements: Fix this line first, then run the program again."
            return {"results": [{**state, "feedback": feedback, "used_tools": [], "rewrite_count": 0,
                                 "verify": {"passed": True, "reason": "The code replied to the syntax error directly; no AI was involved"}}]}
        reviewer = create_agent(model, tools=[check_naming, run_code], system_prompt=REVIEWER_PERSONA)
        messages = [{"role": "user", "content": f"Please review this homework:\n```python\n{code}\n```"}]
        used_tools, rewrite_count = [], 0
        while True:
            result = reviewer.invoke({"messages": messages}, config={"recursion_limit": 25})
            # result includes the full record of the previous round; take only this round's new messages so old tool requests aren't counted twice.
            used_tools += [c["name"] for m in result["messages"][len(messages):] for c in (getattr(m, "tool_calls", None) or [])]
            feedback = result["messages"][-1].text.strip()
            verdict = verify(model, code, feedback)
            if verdict["passed"] or rewrite_count >= 1:      # If verification fails, send it back for at most one rewrite
                break
            rewrite_count += 1
            messages = result["messages"] + [{"role": "user", "content": f"The auditor sent it back: {verdict['reason']}. Fix it and rewrite the feedback."}]
        return {"results": [{**state, "feedback": feedback, "used_tools": used_tools,
                             "rewrite_count": rewrite_count, "verify": verdict}]}

    def confirm_before_send(state: BatchState) -> dict:
        # Sending email is irreversible: stop and hand what would be sent to a person to read
        emails = [{"student": r["student"], "email": r["email"], "verified": r["verify"]["passed"],
                   "feedback": r["feedback"]} for r in state["results"]]
        decision = interrupt({"emails_to_send": emails, "options": SEND_OPTIONS})
        if decision == "Send none":
            return {"decision": decision, "sent_log": ["No emails were sent (decision: Send none)"]}
        return {"decision": decision}

    def should_send(state: BatchState) -> str:
        # Choosing "Send none" ends right away, without entering the send_emails node
        return END if state.get("decision") == "Send none" else "send_emails"

    async def send_emails(state: BatchState) -> dict:
        to_send = [r for r in state["results"]
                   if state.get("decision") == "Send all" or (state.get("decision") == "Send only verified" and r["verify"]["passed"])]
        if not to_send:
            return {"sent_log": [f"No emails were sent (decision: {state.get('decision')})"]}
        tools = await get_mcp_tools(mcp_config)
        log = []
        for r in to_send:
            result = await tools["send_email"].ainvoke({"to": r["email"], "subject": "Homework feedback", "body": r["feedback"]})
            log.append(result_text(result))
        return {"sent_log": log}

    builder = StateGraph(BatchState)
    builder.add_node("fetch_homework", fetch_homework)
    builder.add_node("review_one", review_one)
    builder.add_node("confirm_before_send", confirm_before_send)
    builder.add_node("send_emails", send_emails)
    builder.add_edge(START, "fetch_homework")
    builder.add_conditional_edges("fetch_homework", dispatch, ["review_one"])
    builder.add_edge("review_one", "confirm_before_send")
    builder.add_conditional_edges("confirm_before_send", should_send, ["send_emails", END])
    builder.add_edge("send_emails", END)
    return builder.compile(checkpointer=checkpointer)


async def open_checkpointer(path: Path = CHECKPOINT_PATH):
    """interrupt waits for a person to confirm, so the State must be saved; here it goes into a SQLite file (in the cloud, use Postgres)."""
    import aiosqlite
    from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

    return AsyncSqliteSaver(await aiosqlite.connect(str(path)))


class usage_meter(BaseCallbackHandler):
    """Counts how many times the model is called and adds up the tokens (for Colab's built-in model, the tokens are estimates)."""

    def __init__(self):
        self.count = 0
        self.input_tokens = 0
        self.output_tokens = 0

    def on_llm_end(self, response, **kwargs):
        self.count += 1
        for generations in response.generations:
            for g in generations:
                usage = getattr(getattr(g, "message", None), "usage_metadata", None) or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)


In [ ]:
#@title ▶ Run (Step 1): run the whole diagram once, stopping before the email is sent
thread_id = "midterm-batch-1"  #@param {type:"string"}
batch_scope = "Just Chloe"  #@param ["Just Chloe", "All five"]
max_concurrency = 1  #@param {type:"slider", min:1, max:5, step:1}
homework_repo_transport = "Local socket (stdio)"  #@param ["Local socket (stdio)", "URL (Streamable HTTP; run Exercise 1's second cell first)"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import time
from deploy import multi_agent

url = None
if homework_repo_transport.startswith("URL"):
    port_file = BASE / "deploy" / ".mcp_port"
    if not port_file.exists():
        raise SystemExit("⚠️ The MCP server was not started by URL yet: run Exercise 1's second cell first, or choose \"Local socket (stdio)\"")
    url = f"http://127.0.0.1:{port_file.read_text()}/mcp"

checkpointer = await multi_agent.open_checkpointer()
helper = multi_agent.build_multi_agent_helper(model, mcp_config=multi_agent.homework_repo_config(url), checkpointer=checkpointer,
                                              only_students="Chloe" if batch_scope == "Just Chloe" else None)
config = {"configurable": {"thread_id": thread_id}, "max_concurrency": max_concurrency}
if (await helper.aget_state(config)).values:
    await checkpointer.conn.close()
    raise SystemExit(f"⚠️ thread_id \"{thread_id}\" has already been used: pick a new one (for example \"midterm-batch-2\"), or go straight to Step 2")
show_graph(helper)

meter = multi_agent.usage_meter()
t0 = time.time()
result = await helper.ainvoke({"results": []}, {**config, "callbacks": [meter]})
seconds = time.time() - t0
for r in result["results"]:
    verdict = "✅ passed" if r["verify"]["passed"] else "❌ " + r["verify"]["reason"]
    print(f"[{r['student']}] tools: {', '.join(r['used_tools']) or '(no tool requested; that does not mean the AI was not called)'} | verification: {verdict} | rewritten {r['rewrite_count']} time(s)")
    print("Full feedback:\n" + r["feedback"])
    print("Verification reason:", r["verify"]["reason"])
print(f"\n⏱️ {seconds:.0f} s (up to {max_concurrency} at a time); {meter.count} model call(s), about {meter.input_tokens + meter.output_tokens:,} tokens (estimate)")

if "__interrupt__" in result:
    print("\n⏸️  Stopped before sending, waiting for a person to confirm. Emails ready to send:")
    for mail in result["__interrupt__"][0].value["emails_to_send"]:
        print(f"    → {mail['student']} <{mail['email']}> (verification {'passed' if mail['verified'] else 'not passed'})")
        print(mail["feedback"])
print("Next station in the checkpoint:", (await helper.aget_state(config)).next)
await checkpointer.conn.close()
record(f"2-E Exercise 2: the whole diagram (thread_id={thread_id})",
       {"results": [{k: r[k] for k in ("student", "used_tools", "verify", "rewrite_count")} for r in result["results"]],
        "model_calls": meter.count, "tokens": meter.input_tokens + meter.output_tokens, "seconds": round(seconds)})

In [ ]:
#@title ▶ Run (Step 2): a person has read it and decides whether to send
thread_id = "midterm-batch-1"  #@param {type:"string"}
decision = "Send only verified"  #@param ["Send only verified", "Send all", "Send none"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langgraph.types import Command
from deploy import multi_agent

checkpointer = await multi_agent.open_checkpointer()
helper = multi_agent.build_multi_agent_helper(model, checkpointer=checkpointer)
config = {"configurable": {"thread_id": thread_id}}
if (await helper.aget_state(config)).next != ("confirm_before_send",):
    print(f"thread_id \"{thread_id}\" has no email waiting for confirmation: run Step 1 first, or check whether the thread_id has a typo.")
else:
    result = await helper.ainvoke(Command(resume=decision), config)   # the resume value is what interrupt() returns
    for entry in result["sent_log"]:
        print("📮", entry)
    outbox = BASE / "outbox.txt"
    if outbox.exists():
        print("\nLast lines of outbox.txt:")
        print("\n".join(outbox.read_text(encoding="utf-8").splitlines()[-6:]))
    record(f"2-E Exercise 2: decision \"{decision}\" (thread_id={thread_id})", {"sent_log": result["sent_log"]})
await checkpointer.conn.close()

### 👀 What did you see?

- **First round, done when**: you see Chloe's one piece of feedback, the verification result, the email waiting for confirmation, and the next station. Step 1 only prepares the emails; after you decide in Step 2 with the same thread_id, they are written to the fake outbox.
- Second round: change the scope to "All five", use the new ID "midterm-batch-2" in both steps, and set max_concurrency to 1. Alex has a syntax error, so the code replies directly; the other four each go through review and verification, and a rewrite is verified again.
- Third round (optional): use another new ID, keep "All five", change max_concurrency to 3, and record the seconds, model calls and tokens to compare with the second round. **Parallel is not guaranteed to be faster, and the tokens are not guaranteed to be the same**; the model's answers, the number of rewrites, rate limits and system load all affect the result.
- Each reviewer picks its own tools; the value of the verifying agent is its separate checking job and input, not just a different persona. Passing verification still does not mean the content is absolutely correct; feedback that still fails after a rewrite is still listed, and the default "Send only verified" leaves it out. Alex's syntax reply is produced directly by code and is not handed to the verifying agent; the output says why.
- To start over, use a new thread_id; an ID that has been used is never cleared. Get to where you can explain "waiting for confirmation → a person decides → sent_log" before you move to the API exercise.

---
## Exercise 3 (advanced, optional): Wrap it as an API service

> 📍 **Matching slides**: slide 48 (the API box), 49 "Leaving Colab: three things to replace", 51 "Optional reading: choose a deployment by operational needs", 54 "Waiting for confirmation before sending: interrupt"

In Exercise 2 you pressed ▶ in a notebook. In the cloud, nobody presses a notebook's ▶: a staff member presses "Start review" and "Confirm sending" on a back-office web page,
and that page calls your service over **HTTP** (the same set of rules a browser uses to open a web page).

**The three roles of an API: request, endpoint, response**
- **Request**: the staff member picks "Chloe" on the page and clicks "Start review"; the page sends the data "student: Chloe" to your service over HTTP
- **Endpoint**: your service listens at a URL (in this exercise it is Colab's IP + some port), and when an HTTP request arrives it runs the matching action
- **Response**: when the action finishes, the result (the emails to send, the status, any error message) goes back to the page over HTTP

The next cell writes `deploy/server.py`, which uses FastAPI (a Python package for writing APIs) to open four URLs:

| URL | What it does |
| --- | --- |
| `GET /health` | The cloud platform pings it regularly to check the service is still alive |
| `POST /batch` | Start a batch: fetch homework, review, verify, stop before sending, and return the emails waiting for confirmation |
| `POST /batch/{thread_id}/decision` | After a person confirms: send or don't send |
| `GET /threads/{thread_id}` | Check the progress of a batch |

### 🤔 Prediction question
"Start review" and "Confirm sending" are two different requests, and three days may pass between them. How does the second request know which emails to send?

In [ ]:
%%writefile /content/review-bot/deploy/server.py
"""2-E Exercise 3: wrap the multi-agent Homework Helper as a network service (an API).

Once it is in the cloud, a program like this is what actually runs on the cloud machine:
a staff member's admin web page calls it over HTTP to "start a batch review", and it lists the emails waiting to be sent. When the staff member confirms, the page calls it again over HTTP to "send".
Each batch uses one thread_id to remember its progress: starting and confirming are two different requests, and in between the State is kept in the checkpoint.

To start it on your machine or in a container: uvicorn deploy.server:app --port 8080
Homework repo MCP: if the environment variable HOMEWORK_MCP_URL is set, it connects to that URL; if not, it starts one locally (stdio).
"""

import os

from fastapi import FastAPI
from langgraph.types import Command
from pydantic import BaseModel

from deploy import multi_agent
from llm import get_model

app = FastAPI(title="Homework Helper API")
_helper = None


async def helper():
    """Built the first time someone calls it (the model, the MCP config and the checkpoint are each created only once)."""
    global _helper
    if _helper is None:
        _helper = multi_agent.build_multi_agent_helper(
            get_model(),
            mcp_config=multi_agent.homework_repo_config(os.environ.get("HOMEWORK_MCP_URL")),
            checkpointer=await multi_agent.open_checkpointer(),
        )
    return _helper


class BatchRequest(BaseModel):
    thread_id: str
    max_concurrency: int = 2


class DecisionRequest(BaseModel):
    decision: str   # Send all / Send only verified / Send none


@app.get("/health")
def health() -> dict:
    """The cloud platform hits this URL on a schedule to check that the service is still alive."""
    return {"ok": True}


@app.post("/batch")
async def start_batch(req: BatchRequest) -> dict:
    """Fetch homework from the homework repo, review it in parallel, verify it, then stop before sending and wait for a person to confirm."""
    config = {"configurable": {"thread_id": req.thread_id}, "max_concurrency": req.max_concurrency}
    graph = await helper()
    if (await graph.aget_state(config)).values:
        return {"thread_id": req.thread_id, "error": "This ID already has a checkpoint. Check its progress or start with a new ID; it will not re-review or add results."}
    result = await graph.ainvoke({"results": []}, config)
    pending_confirmation = result["__interrupt__"][0].value if "__interrupt__" in result else None
    return {"thread_id": req.thread_id, "pending_confirmation": pending_confirmation}


@app.post("/batch/{thread_id}/decision")
async def confirm_send(thread_id: str, req: DecisionRequest) -> dict:
    """The decision after a person has looked: continue running with the same thread_id."""
    config = {"configurable": {"thread_id": thread_id}}
    graph = await helper()
    if req.decision not in multi_agent.SEND_OPTIONS:
        return {"thread_id": thread_id, "error": "Please choose a valid send decision"}
    if (await graph.aget_state(config)).next != ("confirm_before_send",):
        return {"thread_id": thread_id, "error": "This thread_id has no emails waiting for confirmation. Check its progress first"}
    result = await graph.ainvoke(Command(resume=req.decision), config)
    return {"thread_id": thread_id, "sent_log": result.get("sent_log", [])}


@app.get("/threads/{thread_id}")
async def get_progress(thread_id: str) -> dict:
    """Look up the checkpoint of a thread_id: whether the State was kept and which station it is stopped at."""
    snap = await (await helper()).aget_state({"configurable": {"thread_id": thread_id}})
    v = snap.values
    return {
        "thread_id": thread_id,
        "has_checkpoint": bool(v),
        "next_station": list(snap.next),
        "reviewed_count": len(v.get("results", [])),
        "decision": v.get("decision"),
        "sent_log": v.get("sent_log"),
    }


In [ ]:
#@title ▶ Run: open the helper as an API service inside Colab
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import socket, threading, time
import requests, uvicorn
from deploy import server

old_server = getattr(sys, "_helper_server", None)       # when this cell is re-run, shut down the one started last time
if old_server:
    old_server.should_exit = True
    time.sleep(1)
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
helper_server = uvicorn.Server(uvicorn.Config(server.app, host="127.0.0.1", port=port, log_level="warning"))
threading.Thread(target=helper_server.run, daemon=True).start()
sys._helper_server = helper_server
for _ in range(50):
    try:
        if requests.get(f"http://127.0.0.1:{port}/health", timeout=1).ok:
            break
    except requests.ConnectionError:
        time.sleep(0.2)
else:
    raise SystemExit("⚠️ The API did not pass the health check. Read the error above and restart the service.")
(BASE / "deploy" / ".port").write_text(str(port))
print(f"✅ The helper API is up: http://127.0.0.1:{port}")
for url in ["GET  /health", "POST /batch", "POST /batch/{thread_id}/decision", "GET  /threads/{thread_id}"]:
    print("   ", url)

In [ ]:
#@title ▶ API Step 1: start a batch, read what is waiting to be sent, and check progress
thread_id = "API-batch-1"  #@param {type:"string"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json, requests

port_file = BASE / "deploy" / ".port"
if not port_file.exists():
    raise SystemExit("⚠️ The service is not running yet: run the previous cell first")
url = f"http://127.0.0.1:{port_file.read_text()}"
try:
    started = requests.post(f"{url}/batch", json={"thread_id": thread_id, "max_concurrency": 2}, timeout=900).json()
except requests.ConnectionError:
    raise SystemExit("⚠️ The service is not running (the runtime may have been restarted): run the previous cell first")

if started.get("error"):
    raise SystemExit(started["error"] + " To confirm the existing emails, use the same ID in the next cell.")
to_send = (started.get("pending_confirmation") or {}).get("emails_to_send", [])
print(f"POST /batch → stopped before sending, {len(to_send)} email(s) waiting for confirmation:")
for mail in to_send:
    print(f"    {mail['student']} <{mail['email']}> verification {'passed' if mail['verified'] else 'not passed'}")
    print(mail["feedback"])
print(f"\nGET /threads/{thread_id} →", json.dumps(requests.get(f"{url}/threads/{thread_id}", timeout=30).json(), ensure_ascii=False))

print("Read what is waiting to be sent first, then choose a sending decision in the next cell; nothing has been sent in this cell.")
record(f"2-E Exercise 3: API start (thread_id={thread_id})", started)

**Pause here**: the last cell only started the review and listed what is waiting to be sent. After you have read it, fill in the same thread_id in the next cell and choose a decision.
This example still writes only to the fake outbox; it is split into two cells to leave room for a person to actually read and decide.

In [ ]:
#@title ▶ API Step 2: after a person has read it, confirm the sending and check progress
thread_id = "API-batch-1"  #@param {type:"string"}
decision = "Send only verified"  #@param ["Send only verified", "Send all", "Send none"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json, requests

port_file = BASE / "deploy" / ".port"
if not port_file.exists():
    raise SystemExit("Start the API service and finish API Step 1 first.")
url = f"http://127.0.0.1:{port_file.read_text()}"
confirmation = requests.post(f"{url}/batch/{thread_id}/decision", json={"decision": decision}, timeout=300).json()
print(f"\nPOST /batch/{thread_id}/decision ({decision}) →")
for entry in confirmation.get("sent_log") or [confirmation.get("error")]:
    print("    📮", entry)
print(f"\nGET /threads/{thread_id} →", json.dumps(requests.get(f"{url}/threads/{thread_id}", timeout=30).json(), ensure_ascii=False))
print("GET /threads/not-started-batch →", json.dumps(requests.get(f"{url}/threads/not-started-batch", timeout=30).json(), ensure_ascii=False))
record(f"2-E Exercise 3: API service (thread_id={thread_id})", {"confirmation": confirmation})

### 👀 What did you see?

- The two requests are tied together by the **thread_id**: the first request runs up to `interrupt()`, saves the State into the checkpoint and answers "pending confirmation", and nothing has been sent yet;
  the second request carries the same thread_id and sends or doesn't send according to the decision. A batch that has not started has no checkpoint to find; an ID that has been used cannot start a review again, so results don't get mixed together.
- Cloud machines can be replaced at any time, so when you go live the checkpoint must live in a database (Postgres), not in a SQLite file on this machine.
- In a real launch, `POST /batch` over 200 submissions takes a long time, so it usually becomes "put it in a task queue and answer right away"; platforms like LangSmith Deployment have this built in.
- `server.py` is the API that really runs on the "own container" path (the `Dockerfile` starts `uvicorn deploy.server:app`). A managed platform instead reads `deploy/app_graph.py:graph`, which `langgraph.json` points to, and uses the platform's built-in threads / runs API; there is no `POST /batch`.

---
## Exercise 4 (advanced, optional): Pre-launch checks, and generating the deployment files

> 📍 **Matching slides**: slide 49 (keys go in Secret Manager), 50 (langgraph.json is the packing slip), 55 "After launch: observable and containable"

The last step: before really going to the cloud, check everything with code, then prepare the files to ship.
The cells below write the files needed for the cloud (**they only generate files; nothing is packaged or deployed**):

| File | What it is |
| --- | --- |
| `langgraph.json` | The packing slip: tells LangSmith Deployment / `langgraph dev` "the graph is in which variable of which file" |
| `deploy/app_graph.py` | The entry point the packing slip points to: the model key and the remote MCP URL are read from environment variables |
| `requirements.txt` | Which packages to install (limited to compatible version ranges; `mcp` and `langchain-mcp-adapters` must be pinned together) |
| `Dockerfile` | The recipe for putting the code, `skills/` and `scripts/` into one container |
| `.dockerignore` | What must not go into the container: `.env`, checkpoints, logs |
| `deploy/preflight.py` | Pre-launch checks: every item is checked by code, not by feel |

`langgraph.json` reads `.env` (used only on your own computer and kept out of git). This notebook does not generate `.env`; create it yourself when you want to use `langgraph dev`. The template is `review-bot/.env.example` in the repo.

### 🤔 Prediction question
There are 10 pre-launch checks. Which do you guess will be ⚠️ (still to handle before going to the cloud)?

In [ ]:
%%writefile /content/review-bot/langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "review_bot": "./deploy/app_graph.py:graph"
  },
  "env": ".env"
}


In [ ]:
%%writefile /content/review-bot/deploy/app_graph.py
"""The entry point that langgraph.json points to: LangSmith Deployment / `langgraph dev` imports the graph in this file.

The deployment platform provides its own checkpointer (for example Postgres), so we don't call open_checkpointer() here.
- Model: WORKSHOP_PROVIDER=gemini; the key GOOGLE_API_KEY is injected by the platform's Secrets
- Homework repo MCP: HOMEWORK_MCP_URL points to the URL of a remote MCP server (if it isn't set, one is started on the same machine over stdio)
"""

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
os.environ.setdefault("WORKSHOP_PROVIDER", "gemini")

from deploy.multi_agent import build_multi_agent_helper, homework_repo_config  # noqa: E402
from llm import get_model  # noqa: E402

graph = build_multi_agent_helper(get_model(), mcp_config=homework_repo_config(os.environ.get("HOMEWORK_MCP_URL")))


In [ ]:
%%writefile /content/review-bot/requirements.txt
# Packages the Homework Helper needs in the cloud (version ranges match the notebooks)
langgraph>=1.1,<2
langchain>=1.2,<2
langchain-google-genai>=4,<5
langgraph-checkpoint-sqlite>=3,<4
# MCP: pin the two together, because mismatched versions fail to import
mcp>=1.20,<2
langchain-mcp-adapters>=0.3,<0.4
fastapi>=0.115
uvicorn>=0.30
# Add this when production switches SQLite to Postgres: langgraph-checkpoint-postgres


In [ ]:
%%writefile /content/review-bot/Dockerfile
# Put the Homework Helper (code + skills/ + scripts/) into one container
# From inside the review-bot/ folder:
#   docker build -t review-bot .
#   docker run -p 8080:8080 -e GOOGLE_API_KEY=your-key -e HOMEWORK_MCP_URL=https://your-remote-mcp-url/mcp review-bot
# Don't write keys into this file: when deploying, the cloud's Secret Manager injects them as environment variables
# If HOMEWORK_MCP_URL is not set, the homework repo MCP is started inside the same container over stdio (for demos)
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Code, handbooks and scripts all go into the image, so the handbooks and the code are always the same version
COPY . .

ENV WORKSHOP_PROVIDER=gemini \
    PORT=8080
CMD ["sh", "-c", "uvicorn deploy.server:app --host 0.0.0.0 --port ${PORT}"]


In [ ]:
%%writefile /content/review-bot/.dockerignore
# Things to keep out of the container: keys, local checkpoints, run logs
.env
.llm.json
checkpoints.sqlite*
deploy/*.sqlite*
deploy/.port
deploy/.mcp_port
deploy/*.log
history.jsonl
outbox.txt
2B_results.json
__pycache__/


In [ ]:
%%writefile /content/review-bot/deploy/preflight.py
"""Exercise 4: pre-launch checks. Every item is checked with code (a sensor), not by gut feeling.

Usage: python -m deploy.preflight (inside the review-bot/ folder)
"""

import os
import re
import sys
from pathlib import Path

BASE = Path(__file__).resolve().parent.parent

KEY_PATTERNS = [
    (r"AIza[0-9A-Za-z\-_]{35}", "Google API key"),
    (r"sk-[A-Za-z0-9_\-]{20,}", "OpenAI / Anthropic style key"),
    (r"(?i)(api[_-]?key|token|password)\s*=\s*['\"][^'\"\s]{8,}['\"]", "password hard-coded in the code"),
]
SUSPICIOUS_PATTERNS = [
    (r"curl[^\n|]*\|\s*(ba)?sh|wget[^\n|]*\|\s*(ba)?sh", "download and run right away (curl ... | bash)"),
    (r"base64\s+(-d|--decode)", "decodes content you can't read"),
    (r"(?i)ignore (all |the )?previous", "tells the AI to ignore earlier instructions"),
    (r"(?i)environment variable|read_env|WORKSHOP_API_KEY|\$[A-Z_]*KEY", "tells the AI to read a key or an environment variable"),
]
text_files = {".py", ".md", ".sh", ".json", ".txt", ".toml", ".yaml", ".yml", ""}
skip = {"__pycache__", ".llm.json", "history.jsonl", ".env.example"}


def _files(root: Path):
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix in text_files and not (set(p.parts) & skip) and p.name not in skip:
            yield p


def _scan(root: Path, patterns) -> list[str]:
    found = []
    for p in _files(root):
        if p.resolve() == Path(__file__).resolve():
            continue  # don't scan itself (these patterns are written in this file on purpose)
        for i, line in enumerate(p.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            for pattern, note in patterns:
                if re.search(pattern, line):
                    found.append(f"{p.relative_to(BASE)} line {i}: {note}")
    return found


def run_checks() -> list[tuple[str, str, str]]:
    """Return [(symbol, item, note)]; the symbol is ✅ / ⚠️ / ❌."""
    results = []

    leaks = _scan(BASE, KEY_PATTERNS)
    results.append(("❌" if leaks else "✅", "No hard-coded keys in the code or handbooks",
               "; ".join(leaks) if leaks else "Keys belong in Secrets / Secret Manager and should be read from environment variables"))

    source = os.environ.get("WORKSHOP_PROVIDER") or "(depends on your choice in 0. Environment setup)"
    has_key = bool(os.environ.get("GOOGLE_API_KEY"))
    results.append(("✅" if has_key else "⚠️", "The model key comes from an environment variable",
               f"WORKSHOP_PROVIDER={source}; GOOGLE_API_KEY {'is set' if has_key else 'is not set (Colab built-in model does not need it; in the cloud you must set it)'}"))

    suspicious = _scan(BASE / "skills", SUSPICIOUS_PATTERNS)
    results.append(("❌" if suspicious else "✅", "Third-party Skill scan (the three habits from 2-C)",
               "; ".join(suspicious) if suspicious else "The simple rules found nothing; that does not mean everything in skills/ is safe"))

    sys.path.insert(0, str(BASE))
    from tools.run_code import run_homework

    blocked = run_homework("import os\nos.listdir('.')").startswith("⛔")
    results.append(("✅" if blocked else "❌", "run_code has a fence", "import os is blocked" if blocked else "import os was NOT blocked!"))

    multi_agent = BASE / "deploy" / "multi_agent.py"
    content = multi_agent.read_text(encoding="utf-8") if multi_agent.exists() else ""
    results.append(("✅" if "recursion_limit" in content else "❌", "The agent has a recursion_limit", "Each review sub-agent runs at most 25 steps"))
    results.append(("⚠️" if "SqliteSaver" in content else "✅", "The checkpointer is durable",
               "SQLite lives only on this one machine; cloud machines get replaced, so switch to PostgresSaver when you launch"))
    has_confirm = "interrupt(" in content
    results.append(("✅" if has_confirm else "❌", "An interrupt comes before the irreversible action (sending email)", "The flow stops and waits for a person before sending" if has_confirm else "No interrupt found"))
    has_verifier = "AUDITOR_PERSONA" in content
    results.append(("✅" if has_verifier else "❌", "There is an independent verification agent", "Review and audit have separate duties and inputs; an independent auditor can still miss errors" if has_verifier else "No verification agent found"))

    url = os.environ.get("HOMEWORK_MCP_URL")
    results.append(("✅" if url else "⚠️", "The homework repo MCP is reached by URL (remote MCP)",
               f"HOMEWORK_MCP_URL={url}" if url else "Right now the MCP server is started on the same machine over stdio; to use it across machines, set a remote MCP URL and configure authorization for your service"))

    missing = [f for f in ("langgraph.json", "requirements.txt", "Dockerfile", ".dockerignore") if not (BASE / f).exists()]
    results.append(("❌" if missing else "✅", "Deployment files are all there", f"Missing: {', '.join(missing)}" if missing else "langgraph.json, requirements.txt, Dockerfile, .dockerignore"))
    return results


def print_check_results() -> bool:
    results = run_checks()
    for symbol, item, note in results:
        print(f"{symbol} {item}\n     {note}")
    failed = sum(1 for r in results if r[0] == "❌")
    print(f"\n{'🚫 ' + str(failed) + ' item(s) failed. Fix them before launch' if failed else '🎉 No ❌. Handle the ⚠️ items before launch'}")
    return failed == 0


if __name__ == "__main__":
    sys.exit(0 if print_check_results() else 1)


In [ ]:
#@title ▶ Run: pre-launch checks
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from deploy import preflight

passed = preflight.print_check_results()
record("2-E Exercise 4: pre-launch checks", [{"result": a, "item": b, "note": c} for a, b, c in preflight.run_checks()])

### 👀 What did you see?

- ✅ only means it passed this example's check: for example, no listed key pattern was matched, a simple scan found none of the listed suspicious words, `run_code` has the execution limits from the class example, the agent has a `recursion_limit`,
  there is an `interrupt` before sending email, there is a separate verifying agent, and the deployment files are all there.
- ⚠️ marks what still needs handling before the cloud:
  - **Key**: this Gemini deployment example needs a valid API key, kept in Secret Manager (Colab's 🔑 "Secrets" on the left is a small version of it).
  - **checkpointer**: SQLite exists only on this one machine; when going live, switch to PostgresSaver.
  - **MCP**: right now the server is started on the same machine over stdio; across machines you can switch to a remote MCP URL (`HOMEWORK_MCP_URL`), authorized the way that service sets up; OAuth is one such way.
- To try 2-C again: put the problem handbook `detailed-feedback` into `skills/`, run the check again, and see whether "third-party Skill scan" turns into ❌.

### When you really deploy (reference for after class; it costs money, and the commands follow each platform's official documentation)

```bash
# Inside the review-bot/ folder
# 0. Copy .env.example to .env and fill in your own GOOGLE_API_KEY (.env stays out of git)
# 1. Get it working locally first (needs pip install "langgraph-cli[inmem]")
langgraph dev

# 2a. Your own container: build an image and put it on a service like Cloud Run; inject the key with Secret Manager
docker build -t review-bot .
gcloud run deploy review-bot --source . --set-secrets=GOOGLE_API_KEY=review-bot-key:latest

# 2b. Managed: push this folder to GitHub, and in LangSmith Deployment pick this repo to deploy (it reads langgraph.json)
```

A real deployment also needs the runtime to be managed: **every step visible (trace, eval, cost), and permissions properly limited (least privilege, sandbox, review of Skills, keys, human confirmation).**

## Wrap-up: save your work and hand it in

- 💾 **Save to Google Drive**: save a copy of `/content/review-bot/` (your edited handbooks, code, and State records) to your own Google Drive.
- ⬇️ **Download the zip**: download it straight to your computer.
- 📸 **The last cell**: prints the results the experiment cells saved. Pick the relevant records for a screenshot, then add your observations and reasons as the assignment asks.

These are the results the experiment cells saved through `record()`, not every intermediate state of every run.
Records in the same runtime pile up, so they may include experiments from other notebooks; use the printed titles to tell them apart.

A temporary browser disconnect does not necessarily wipe your files; files under `/content/` are lost only when the runtime is deleted or recycled.
To keep your handbooks, code, and checkpoint, back them up first. The notebook itself is saved with the menu "File → Save a copy in Drive". The two kinds of saving do different jobs.
When you come back, first check that the runtime is still there. If it was reset, re-run "0. Setup" and this part's setup cell, then restore the files you backed up as needed.

In [ ]:
#@title 💾 Save to Google Drive (an authorization window will pop up)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
save_to_drive()

In [ ]:
#@title ⬇️ Download the zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
download_zip()

In [ ]:
#@title 📸 Print the recorded experiment results (pick the relevant records to hand in)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
print_history()